# Unified Validation Notebook

Use this notebook as a thin front-end for `experiments/validate_methods.py`.

It supports:
- RLlib checkpoint validation
- fixed-time validation
- static max-pressure validation
- optional per-seed recording through the CLI

The notebook does not reimplement evaluation logic. It builds a CLI command, runs it, and then reads the saved artifacts.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "sumo_rl").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current working directory.")


ROOT = find_repo_root()
NOTEBOOK_DIR = ROOT / "experiments"
PYTHON_EXE = ROOT / ".venv" / "Scripts" / "python.exe"
if not PYTHON_EXE.exists():
    PYTHON_EXE = Path(sys.executable)

VALIDATION_CLI = ROOT / "experiments" / "validate_methods.py"

print(f"Repo root: {ROOT}")
print(f"Python executable: {PYTHON_EXE}")
print(f"Validation CLI: {VALIDATION_CLI}")
print(f"pandas available: {pd is not None}")

## Configure the validation run

Set the controller and the corresponding inputs:
- `rllib`: set `RUN_DIR` and optionally `CHECKPOINT_PATH` or `CHECKPOINT_SELECTOR`
- `fixed_time`: set `SCENARIO`
- `static_max_pressure`: set `SCENARIO`


In [ ]:
# Required high-level selection.
CONTROLLER = "fixed_time"  # one of: "rllib", "fixed_time", "static_max_pressure"

# RLlib inputs.
RUN_DIR = None
CHECKPOINT_PATH = None
CHECKPOINT_DIR = None
CHECKPOINT_SELECTOR = "best"  # "best" or "latest"

# Static-controller inputs.
SCENARIO = "resco_grid4x4"
CONFIG_NAME = None
EXTRA_OVERRIDES = []  # e.g. ["env.kwargs.num_seconds=600"]

# Shared validation inputs.
SEEDS = [1]
PARALLEL_WORKERS = 1
METRICS_PROFILE = "thesis-default"  # or "full"
SHOW_TERMINAL_TABLE = "compact"  # or "compact+fairness", "full"
DISABLE_COMPLETED_VIEW = False

# Optional recording.
RECORD_SEED = None
RECORD_OUTPUT_DIR = None
USE_GUI = False
WIDTH = 1600
HEIGHT = 900
FPS = 10
FRAME_SKIP = 1

# Optional output location.
OUTPUT_DIR = None

In [ ]:
def build_validation_command() -> list[str]:
    if CONTROLLER not in {"rllib", "fixed_time", "static_max_pressure"}:
        raise ValueError(f"Unsupported CONTROLLER: {CONTROLLER}")

    command = [
        str(PYTHON_EXE),
        str(VALIDATION_CLI),
        "--controller",
        CONTROLLER,
        "--metrics-profile",
        str(METRICS_PROFILE),
        "--show-terminal-table",
        str(SHOW_TERMINAL_TABLE),
        "--parallel-workers",
        str(int(PARALLEL_WORKERS)),
        "--width",
        str(int(WIDTH)),
        "--height",
        str(int(HEIGHT)),
        "--fps",
        str(int(FPS)),
        "--frame-skip",
        str(int(FRAME_SKIP)),
    ]

    if SEEDS:
        command.append("--seeds")
        command.extend(str(int(seed)) for seed in SEEDS)

    if OUTPUT_DIR:
        command.extend(["--output-dir", str(Path(OUTPUT_DIR).expanduser())])
    if DISABLE_COMPLETED_VIEW:
        command.append("--disable-completed-view")
    if RECORD_SEED is not None:
        command.extend(["--record-seed", str(int(RECORD_SEED))])
    if RECORD_OUTPUT_DIR:
        command.extend(["--record-output-dir", str(Path(RECORD_OUTPUT_DIR).expanduser())])
    if USE_GUI:
        command.append("--use-gui")

    if CONTROLLER == "rllib":
        if not RUN_DIR:
            raise ValueError("Set RUN_DIR for CONTROLLER='rllib'.")
        command.extend(["--run-dir", str(Path(RUN_DIR).expanduser())])
        if CHECKPOINT_PATH:
            command.extend(["--checkpoint-path", str(Path(CHECKPOINT_PATH).expanduser())])
        elif CHECKPOINT_DIR:
            command.extend(["--checkpoint-dir", str(Path(CHECKPOINT_DIR).expanduser())])
            command.extend(["--checkpoint-selector", str(CHECKPOINT_SELECTOR)])
        else:
            command.extend(["--checkpoint-selector", str(CHECKPOINT_SELECTOR)])
    else:
        if SCENARIO:
            command.extend(["--scenario", str(SCENARIO)])
        if CONFIG_NAME:
            command.extend(["--config-name", str(CONFIG_NAME)])
        for override in EXTRA_OVERRIDES:
            command.extend(["--override", str(override)])

    return command


COMMAND = build_validation_command()
print("Command:")
print(" ".join(f'\"{part}\"' if " " in part else part for part in COMMAND))

## Run the CLI

This cell launches the script and then discovers the output directory from the terminal output.

In [ ]:
result = subprocess.run(
    COMMAND,
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

stdout_text = result.stdout or ""
stderr_text = result.stderr or ""

print(stdout_text)
if stderr_text.strip():
    print("--- STDERR ---")
    print(stderr_text)

if result.returncode != 0:
    raise RuntimeError(f"Validation CLI failed with exit code {result.returncode}.")

OUTPUT_DIR_RESOLVED = None
for line in stdout_text.splitlines():
    prefix = "Validation output dir: "
    if line.startswith(prefix):
        OUTPUT_DIR_RESOLVED = Path(line[len(prefix):].strip())
        break

if OUTPUT_DIR_RESOLVED is None:
    if OUTPUT_DIR:
        OUTPUT_DIR_RESOLVED = Path(OUTPUT_DIR).expanduser().resolve()
    else:
        raise RuntimeError("Could not determine the validation output directory from CLI output.")

print(f"Resolved output dir: {OUTPUT_DIR_RESOLVED}")

## Load the saved artifacts

In [ ]:
summary_path = OUTPUT_DIR_RESOLVED / "summary.json"
seed_rows_path = OUTPUT_DIR_RESOLVED / "seed_rows.json"
terminal_summary_path = OUTPUT_DIR_RESOLVED / "terminal_summary.txt"
errors_path = OUTPUT_DIR_RESOLVED / "errors.json"

summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
seed_rows = json.loads(seed_rows_path.read_text(encoding="utf-8")) if seed_rows_path.exists() else []
terminal_summary = terminal_summary_path.read_text(encoding="utf-8") if terminal_summary_path.exists() else ""
errors = json.loads(errors_path.read_text(encoding="utf-8")) if errors_path.exists() else {}

print("Terminal summary:")
print(terminal_summary)

if errors:
    print("Errors:")
    print(json.dumps(errors, indent=2))

In [ ]:
if pd is not None and seed_rows:
    seed_rows_df = pd.DataFrame(seed_rows)
    display(seed_rows_df)
else:
    seed_rows_df = None
    print(seed_rows)

In [ ]:
if pd is not None and summary:
    summary_df = pd.DataFrame([summary])
    display(summary_df)
else:
    summary_df = None
    print(summary)

## Inspect generated files

In [ ]:
for path in sorted(OUTPUT_DIR_RESOLVED.rglob("*")):
    print(path.relative_to(OUTPUT_DIR_RESOLVED))